In [1]:
"""
Teacher/Student scaling law experiment — Table 1 architecture
(Sharma & Kaplan, arXiv:2004.10802)


This version:
- removes custom phase schedule
- uses standard Keras training with (epochs, batch_size)
- stores training data in the main directory, overwriting each d
- uses MLE (Levina-Bickel) intrinsic dimension estimator instead of TwoNN
- records intermediate test MSE at per-phase checkpoint intervals for every width
- possibility to show a dedicated error-curve plot for each width right after it finishes training
"""


import os
import random
import gc
from pathlib import Path


import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.spatial import cKDTree



# ──────────────────────────────────────────────
# Global seed & determinism
# ──────────────────────────────────────────────
GLOBAL_SEED = 1


num_cores = os.cpu_count()
tf.config.threading.set_intra_op_parallelism_threads(num_cores)
tf.config.threading.set_inter_op_parallelism_threads(num_cores)


os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)
tf.keras.utils.set_random_seed(GLOBAL_SEED)
tf.config.experimental.enable_op_determinism()



# ──────────────────────────────────────────────
# Paths
# ──────────────────────────────────────────────
REPO_ROOT = Path(".").resolve()
MODEL_ROOT = REPO_ROOT / "models_std4_train"
MODEL_ROOT.mkdir(exist_ok=True)




# ──────────────────────────────────────────────
# Fixed experiment parameters
# ──────────────────────────────────────────────
D_INPUT     = 20
LATENT_DIMS = [2, 4, 6, 8, 10 ,12]


N_TEST      = 20_000
WIDTHS      = [8, 10, 12,  16, 18, 20, 24, 28, 32, 36, 40, 46, 52, 58, 64, 72, 80, 96, 128, 168]
WIDTH_F     = 1
DEPTH_T     = 2
DEPTH_S     = 2


SCHEDULE = [
    ( 5_000, 500,  0.01),
    ( 2_000, 1000,  0.005),
    ( 2_000, 2000, 0.002),
    ( 500, 5000, 0.001),
]


TEACHER_WIDTH = 300


# ──────────────────────────────────────────────
# Checkpoint intervals: one entry per phase in SCHEDULE.
# A checkpoint (test-MSE evaluation) is recorded every
# `interval` local steps within that phase.
# ──────────────────────────────────────────────
CHECKPOINT_INTERVALS = [1000, 400, 400, 100]   # must match len(SCHEDULE)


# ──────────────────────────────────────────────
# Plotting option: show a dedicated per-width error curve
# right after that width finishes training.
# ──────────────────────────────────────────────
ERROR_CHECK      = False 
PLOT_LIVE_CURVES = False   # set False to disable plotting entirely
SHOW_PLOTS       = False   # if True, plt.show() blocks until the window is closed
                           # if False, plots are only saved to disk (non-blocking)


# ──────────────────────────────────────────────
# MLE intrinsic dimension hyperparameter
# ──────────────────────────────────────────────
MLE_K = 10   # number of nearest neighbors used in Levina-Bickel MLE




# ──────────────────────────────────────────────
# Helper functions
# ──────────────────────────────────────────────
def sample_inputs(n: int, d_latent: int, W_EMBED) -> np.ndarray:
    z = np.random.randn(n, d_latent).astype(np.float32)
    x_lin = z @ W_EMBED / np.sqrt(d_latent)
    return x_lin.astype(np.float32)


def make_and_save_fixed_dataset(schedule, d_latent, W_EMBED, teacher, save_dir):
    """
    For a given d_latent, create a fixed training dataset once and save to disk
    in save_dir using memory-mapped .dat files.

    Saves:
      train_inputs_d{d_latent}.dat  : shape (N_total, D_INPUT)
      train_targets_d{d_latent}.dat : shape (N_total, WIDTH_F)
    Returns:
      total_examples, phase_offsets, inputs_fname, targets_fname
    """
    total_examples = 0
    phase_sizes = []
    for (n_steps, batch_size, _) in schedule:
        size = n_steps * batch_size
        phase_sizes.append(size)
        total_examples += size

    inputs_path  = (save_dir / f"train_inputs_d{d_latent}.dat").resolve()
    targets_path = (save_dir / f"train_targets_d{d_latent}.dat").resolve()

    X_all = np.memmap(str(inputs_path), dtype="float32", mode="w+",
                      shape=(total_examples, D_INPUT))
    Y_all = np.memmap(str(targets_path), dtype="float32", mode="w+",
                      shape=(total_examples, WIDTH_F))

    phase_offsets = []
    idx = 0
    for ((n_steps, batch_size, _), phase_size) in zip(schedule, phase_sizes):
        phase_offsets.append(idx)
        remaining = phase_size
        while remaining > 0:
            b = min(batch_size, remaining)
            x = sample_inputs(b, d_latent, W_EMBED)
            y = teacher(x, training=False).numpy()
            X_all[idx:idx + b] = x
            Y_all[idx:idx + b] = y
            idx += b
            remaining -= b

    X_all.flush()
    Y_all.flush()
    del X_all
    del Y_all

    return total_examples, phase_offsets, str(inputs_path), str(targets_path)



def build_mlp(width: int, width_f: int, depth: int,
              name: str = "net") -> tf.keras.Model:
    inp = tf.keras.Input(shape=(D_INPUT,))
    x   = inp
    for _ in range(depth):
        x = tf.keras.layers.Dense(
            width,
            activation="relu",
            kernel_initializer="he_normal"
        )(x)
    out = tf.keras.layers.Dense(
        width_f,
        kernel_initializer="he_normal"
    )(x)
    return tf.keras.Model(inp, out, name=name)



def train_from_disk_with_schedule(model, schedule, X_all, Y_all, phase_offsets,
                                   x_test=None, y_test=None,
                                   checkpoint_intervals=None, ERROR_CHECK= False):
    """
    Train `model` using a fixed on-disk dataset X_all, Y_all and the given schedule.
    All students with the same d_latent share X_all, Y_all.

    checkpoint_intervals : list matching `schedule`, one interval per phase.
                            A checkpoint (test-MSE evaluation) is recorded
                            every `interval` local steps within that phase,
                            plus always at the end of each phase.
    Returns: history dict {global_step: mse}
    """
    history = {}
    global_step = 0

    for phase_idx, (n_steps, batch_size, lr) in enumerate(schedule):
        interval = checkpoint_intervals[phase_idx] if checkpoint_intervals else None

        ckpt_msg = f"  (checkpoint every {interval} steps)" if interval else ""
        print(f"    Phase {phase_idx + 1}: {n_steps:>7,d} steps  "
              f"batch={batch_size:>4d}  LR={lr}{ckpt_msg}")

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
            loss="mse"
        )

        start = phase_offsets[phase_idx]
        for local_step in range(n_steps):
            offset = start + local_step * batch_size
            x_batch = X_all[offset:offset + batch_size]
            y_batch = Y_all[offset:offset + batch_size]
            model.train_on_batch(x_batch, y_batch)

            global_step += 1

            if ERROR_CHECK and interval and (local_step + 1) % interval == 0:
                if x_test is not None and y_test is not None:
                    mse_ckpt = float(model.evaluate(x_test, y_test, verbose=0))
                    history[global_step] = mse_ckpt
                    print(f"      [phase {phase_idx+1}, local step {local_step+1:>6,d}, "
                          f"global step {global_step:>7,d}] test MSE = {mse_ckpt:.4e}")

        # always record an end-of-phase checkpoint, in case it wasn't hit above
        if ERROR_CHECK and x_test is not None and y_test is not None and global_step not in history:
            mse_ckpt = float(model.evaluate(x_test, y_test, verbose=0))
            history[global_step] = mse_ckpt
            print(f"      [end of phase {phase_idx+1}, global step {global_step:>7,d}] "
                  f"test MSE = {mse_ckpt:.4e}")

    return history


def plot_width_curve(history, width, d_latent, save_dir, show=True):
    """
    Create, save, and (optionally) display a dedicated error-curve plot
    for a single width, called right after that width finishes training.
    """
    steps = sorted(history.keys())
    mses  = [history[s] for s in steps]

    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.plot(steps, mses, "o-", color="steelblue")
    ax.set_xlabel("Training step")
    ax.set_ylabel("Test MSE (log scale)")
    ax.set_yscale("log")
    ax.set_title(f"d = {d_latent},  width = {width}")
    ax.grid(True, which="both", ls="--", alpha=0.3)
    fig.tight_layout()

    save_path = save_dir / f"error_curve_d{d_latent}_w{width}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)

    return save_path


def power_law_with_floor(N, C0, C1, alpha):
    alpha = np.clip(alpha, -10, 10)
    return C0 + C1 * N ** (-alpha)



# ──────────────────────────────────────────────
# MLE (Levina-Bickel) intrinsic dimension estimator
# replaces twonn_id
# ──────────────────────────────────────────────
def mle_id(X, n_samples=12000, seed=0, k=MLE_K):
    """
    Levina-Bickel maximum-likelihood estimator of intrinsic dimension.
    X : (n_points, n_features) array of representations / activations
    n_samples : subsample size for tractability
    k : number of nearest neighbors used in the local dimension estimate
    """
    rng = np.random.default_rng(seed)

    N_total = X.shape[0]
    if N_total > n_samples:
        idx = rng.choice(N_total, size=n_samples, replace=False)
        X = X[idx]

    n = X.shape[0]
    k_eff = min(k, n - 1)
    if k_eff < 2:
        raise RuntimeError("Too few points for MLE intrinsic dimension estimate.")

    tree = cKDTree(X)
    dist, _ = tree.query(X, k=k_eff + 1)   # includes self-distance at column 0
    dist = np.maximum(dist[:, 1:], 1e-12)  # drop self, avoid log(0)

    T_k    = dist[:, -1:]                  # distance to k-th neighbor
    ratios = np.log(T_k / dist)[:, :-1]    # j = 1..k-1
    local_est = (k_eff - 1) / np.sum(ratios, axis=1)

    finite = np.isfinite(local_est) & (local_est > 0)
    local_est = local_est[finite]
    if len(local_est) < 10:
        raise RuntimeError("Too few valid points for MLE intrinsic dimension estimate.")

    d_hat = 1.0 / np.mean(1.0 / local_est)
    return float(d_hat)



def sample_activation_cloud(model, d_latent, n_points=12000, batch_size=2048):
    acts_list = []
    remaining = n_points

    hidden_model = tf.keras.Model(
        inputs=model.input,
        outputs=model.layers[-2].output
    )

    while remaining > 0:
        b = min(batch_size, remaining)
        x = sample_inputs(b, d_latent, W_EMBED)
        h = hidden_model(x, training=False).numpy()
        acts_list.append(h)
        remaining -= b

    return np.concatenate(acts_list, axis=0)



# ──────────────────────────────────────────────
# Main sweep
# ──────────────────────────────────────────────
all_results = {}


for D_LATENT in LATENT_DIMS:
    print("\n" + "=" * 70)
    print(f"  d = {D_LATENT}  (theory alpha = {4.0/D_LATENT:.3f})")
    print("=" * 70)


    save_dir = MODEL_ROOT / f"d{D_LATENT}"
    save_dir.mkdir(parents=True, exist_ok=True)


    # ── Teacher ────────────────────────────────
    teacher = build_mlp(
        width=TEACHER_WIDTH, width_f=WIDTH_F,
        depth=DEPTH_T, name=f"teacher_d{D_LATENT}"
    )
    teacher.trainable = False

    print(f"Teacher: [20, {TEACHER_WIDTH}, {TEACHER_WIDTH}, {WIDTH_F}]  "
          f"({teacher.count_params():,} params)")

    teacher.save(save_dir / "teacher.keras")
    print(f"  → teacher saved to {save_dir / 'teacher.keras'}")


    # ── Datasets ───────────────────────────────
    W_EMBED = np.random.randn(D_LATENT, D_INPUT).astype(np.float32)

    total_train, phase_offsets, inputs_fname, targets_fname = make_and_save_fixed_dataset(
        SCHEDULE, D_LATENT, W_EMBED, teacher, save_dir
    )

    X_all = np.memmap(inputs_fname, dtype="float32", mode="r",
                      shape=(total_train, D_INPUT))
    Y_all = np.memmap(targets_fname, dtype="float32", mode="r",
                      shape=(total_train, WIDTH_F))

    x_test  = sample_inputs(N_TEST,  D_LATENT, W_EMBED)
    y_test  = teacher(x_test,  training=False).numpy()

    d_intrinsic_T = mle_id(
        sample_activation_cloud(teacher, D_LATENT, n_points=8000),
        n_samples=8000,
        seed=GLOBAL_SEED,
        k=MLE_K,
    )

    results       = []
    d_intrinsic_S = []
    checkpoint_histories = {}   # {width: {global_step: mse}}
    prev_test_mse = None


    # ── Students ───────────────────────────────
    for w in WIDTHS:
        student = build_mlp(
            width=w, width_f=WIDTH_F,
            depth=DEPTH_S, name=f"student_d{D_LATENT}_w{w}"
        )
        print(f"  Training student [20, {w:>3d}, {w:>3d}, {WIDTH_F}]  "
              f"(N ≈ {student.count_params():>7,d} params)")

        history = train_from_disk_with_schedule(
            student, SCHEDULE, X_all, Y_all, phase_offsets,
            x_test=x_test, y_test=y_test,
            checkpoint_intervals=CHECKPOINT_INTERVALS,
        )
        checkpoint_histories[w] = history

        test_mse = float(student.evaluate(x_test, y_test, verbose=0))
        N        = student.count_params()

        results.append((N, test_mse))

        d_intrinsic_S.append(
            mle_id(
                sample_activation_cloud(student, D_LATENT, n_points=8000),
                n_samples=8000,
                seed=GLOBAL_SEED,
                k=MLE_K,
            )
        )

        save_path = save_dir / f"student_w{w}.keras"
        student.save(save_path)

        print(f"    → final test MSE = {test_mse:.4e}   N = {N:,}")
        print(f"    → saved to {save_path}")

        # Dedicated per-width error-curve plot, shown/saved at the end of this iteration
        if PLOT_LIVE_CURVES:
            plot_path = plot_width_curve(history, w, D_LATENT, save_dir, show=SHOW_PLOTS)
            print(f"    → error curve saved to {plot_path}")

        print()

        prev_test_mse = test_mse


    # Release memmaps for this d before moving to next d
    del X_all
    del Y_all


    # ── Scaling law fit ────────────────────────
    alpha_theory = 4.0 / D_LATENT

    N_vals = np.array([r[0] for r in results], dtype=float)
    L_vals = np.array([r[1] for r in results], dtype=float)

    p0 = [L_vals.min() * 0.9, L_vals.max() - L_vals.min(), alpha_theory]

    try:
        popt, _ = curve_fit(
            power_law_with_floor, N_vals, L_vals, p0=p0,
            bounds=([0, 0, 0], [np.inf, np.inf, 10]),
            maxfev=10_000,
        )
        C0_est, C1_est, alpha_est = popt
    except RuntimeError:
        print("  ⚠  curve_fit did not converge; storing NaNs.")
        C0_est, C1_est, alpha_est = np.nan, np.nan, np.nan


    print("─── Scaling law fit ─────────────────────")
    print(f"  C0     = {C0_est:.4e}")
    print(f"  C1     = {C1_est:.4e}")
    print(f"  alpha  = {alpha_est:.4f}  (theory: {alpha_theory:.4f},"
          f"  ratio: {alpha_est/alpha_theory:.3f})")
    print()


    # ── Flatten checkpoint histories for saving ────
    ckpt_widths, ckpt_steps, ckpt_mses = [], [], []
    for w, hist in checkpoint_histories.items():
        for step, mse in hist.items():
            ckpt_widths.append(w)
            ckpt_steps.append(step)
            ckpt_mses.append(mse)
    ckpt_widths = np.array(ckpt_widths, dtype=int)
    ckpt_steps  = np.array(ckpt_steps, dtype=int)
    ckpt_mses   = np.array(ckpt_mses, dtype=float)


    np.savez(
        save_dir / "results.npz", widths=WIDTHS,
        N_vals=N_vals, L_vals=L_vals,
        C0=C0_est, C1=C1_est,
        alpha=alpha_est, alpha_th=alpha_theory,
        d_latent=D_LATENT,
        d_intrinsic_T=d_intrinsic_T,
        d_intrinsic_S=d_intrinsic_S,
        checkpoint_intervals=CHECKPOINT_INTERVALS,
        ckpt_widths=ckpt_widths,
        ckpt_steps=ckpt_steps,
        ckpt_mses=ckpt_mses,
    )
    print(f"  → results saved to {save_dir / 'results.npz'}")


    all_results[D_LATENT] = {
        "schedule":              SCHEDULE,
        "N_vals":                N_vals,
        "L_vals":                L_vals,
        "C0":                    C0_est,
        "C1":                    C1_est,
        "alpha":                 alpha_est,
        "alpha_th":              alpha_theory,
        "d intrinsic_T":         d_intrinsic_T,
        "d intrinsic_S":         d_intrinsic_S,
        "checkpoint_histories":  checkpoint_histories,
    }


# ──────────────────────────────────────────────
# Final summary
# ──────────────────────────────────────────────
print("\n" + "=" * 70)
print("SUMMARY")
print(f"{'d':>4}  {'alpha_fit':>10}  {'4/d':>8}  {'ratio':>7} {'4/d intrinsic_T':>20}")
print("-" * 60)
for d, r in all_results.items():
    ratio = r["alpha"] / r["alpha_th"] if r["alpha_th"] else float("nan")
    print(f"{d:>4}  {r['alpha']:>10.4f}  {r['alpha_th']:>8.4f}  {ratio:>7.3f} {4/r['d intrinsic_T']:>17.3f}")


  d = 2  (theory alpha = 2.000)
Teacher: [20, 300, 300, 1]  (96,901 params)
  → teacher saved to C:\Users\Priet005\OneDrive - Universiteit Utrecht\Documents\My repo\ML and information theory\Error vs dimension\models_std4_train\d2\teacher.keras
  Training student [20,   8,   8, 1]  (N ≈     249 params)
    Phase 1:   5,000 steps  batch= 500  LR=0.01  (checkpoint every 1000 steps)
    Phase 2:   2,000 steps  batch=1000  LR=0.005  (checkpoint every 400 steps)
    Phase 3:   2,000 steps  batch=2000  LR=0.002  (checkpoint every 400 steps)
    Phase 4:     500 steps  batch=5000  LR=0.001  (checkpoint every 100 steps)
    → final test MSE = 9.4711e-05   N = 249
    → saved to C:\Users\Priet005\OneDrive - Universiteit Utrecht\Documents\My repo\ML and information theory\Error vs dimension\models_std4_train\d2\student_w8.keras

  Training student [20,  10,  10, 1]  (N ≈     331 params)
    Phase 1:   5,000 steps  batch= 500  LR=0.01  (checkpoint every 1000 steps)
    Phase 2:   2,000 steps  ba